# Chapter 11 Training Deep Neural Networks
## Section 3 Faster Optimizers
### Section 3.6 Learning Rate Scheduling

In [1]:
import time
from pathlib import Path
from typing import Callable, List

import numpy as np
import tensorflow.keras as keras

#### Theory
##### Power scheduling

In [2]:
optimizer = keras.optimizers.SGD(lr=0.01, decay=1e-4)

##### Exponential scheduling

Option 1

In [3]:
def exponential_decay_generator(lr: float = 0.01, s: int = 20) -> Callable:
    def exponential_decay_fn(epoch: int) -> float:
        return lr * 0.1 ** (epoch / s)

    return exponential_decay_fn

In [4]:
lr_scheduler = keras.callbacks.LearningRateScheduler(exponential_decay_generator())
# history = model.fit(X_train_scaled, y_train, [...], callbacks[lr_scheduler])

Option2

In [5]:
epochs = 20
batch = 32
X_train = "spam"
steps = 20 * len(X_train) // 32
lr = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.01, decay_steps=steps, decay_rate=0.1
)
optimizer = keras.optimizers.SGD(learning_rate=lr)

##### Piecewise constant scheduling

In [6]:
def piecewise_constant_fn(epoch: int) -> float:
    if epoch < 5:
        return 0.01
    elif epoch < 15:
        return 0.005
    else:
        return 0.001

##### Performance scheduling

In [7]:
lr_scheduler = keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)

#### Experiment

Comparison between
- No scheduling
- Exponential scheduling
- Performance scheduling
- 1cycle scheduling

Dataset

In [8]:
# Load dataset
fashion_mnist = keras.datasets.fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()
class_names = np.array(
    [
        "T-shirt/top",
        "Trouser",
        "Pullover",
        "Dress",
        "Coat",
        "Sandal",
        "Shirt",
        "Sneaker",
        "Bag",
        "Ankle boot",
    ]
)

# Normalize data
X_train, X_valid = X_train_full[:-5000] / 255.0, X_train_full[-5000:] / 255.0
y_train, y_valid = y_train_full[:-5000], y_train_full[-5000:]

Model

In [9]:
model = keras.Sequential(
    [
        keras.layers.Flatten(input_shape=(28, 28)),
        keras.layers.Dense(300, activation="selu", kernel_initializer="lecun_normal"),
        keras.layers.Dense(100, activation="selu", kernel_initializer="lecun_normal"),
        keras.layers.Dense(len(class_names), activation="softmax"),
    ]
)
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
flatten (Flatten)            (None, 784)               0         
_________________________________________________________________
dense (Dense)                (None, 300)               235500    
_________________________________________________________________
dense_1 (Dense)              (None, 100)               30100     
_________________________________________________________________
dense_2 (Dense)              (None, 10)                1010      
Total params: 266,610
Trainable params: 266,610
Non-trainable params: 0
_________________________________________________________________


Train

In [10]:
%load_ext tensorboard
%tensorboard --logdir=./my_logs --port=6006

Launching TensorBoard...

In [11]:
epochs = 50
batch = 32

steps = epochs * len(X_train) // 32

In [12]:
class Experiment:
    def __init__(
        self,
        model: keras.Model,
        optimizer: keras.optimizers.Optimizer,
        epochs: int,
        callbacks: List[keras.callbacks.Callback],
    ):
        self.model: keras.Model = keras.models.clone_model(model)
        self.optimizer: keras.optimizers.Optimizer = optimizer
        self.epochs: int = epochs
        self.callbacks: List[keras.callbacks.Callback] = callbacks
        self.dir_log_run: str = ""

        self.setup_log_dir()
        self.compile()

    def setup_log_dir(self):
        run_id = time.strftime("run_%Y_%m_%d-%H_%M_%S")
        self.dir_log_run = str(Path() / "my_logs" / run_id)

    def compile(self) -> None:
        self.model.compile(
            loss="sparse_categorical_crossentropy",
            optimizer=keras.optimizers.SGD(),
            metrics=["accuracy"],
        )

    def fit(self, X_train, y_train, X_valid, y_valid) -> keras.callbacks.History:
        tensorboard_cb = keras.callbacks.TensorBoard(self.dir_log_run)
        return self.model.fit(
            X_train,
            y_train,
            epochs=self.epochs,
            validation_data=(X_valid, y_valid),
            callbacks=[tensorboard_cb] + self.callbacks,
        )

In [13]:
# No scheduling
exp_none = Experiment(
    model=model,
    optimizer=keras.optimizers.SGD(),
    epochs=epochs,
    callbacks=[],
)
history_none = exp_none.fit(
    X_train=X_train, y_train=y_train, X_valid=X_valid, y_valid=y_valid
)

Epoch 1/50
1719/1719 [==============================] - 3s 2ms/step - loss: 0.7601 - accuracy: 0.7437 - val_loss: 0.4723 - val_accuracy: 0.8280
Epoch 2/50
1719/1719 [==============================] - 2s 1ms/step - loss: 0.4571 - accuracy: 0.8389 - val_loss: 0.4315 - val_accuracy: 0.8454
Epoch 3/50
1719/1719 [==============================] - 2s 1ms/step - loss: 0.4192 - accuracy: 0.8518 - val_loss: 0.4193 - val_accuracy: 0.8436
Epoch 4/50
1719/1719 [==============================] - 2s 1ms/step - loss: 0.3928 - accuracy: 0.8618 - val_loss: 0.3945 - val_accuracy: 0.8574
Epoch 5/50
1719/1719 [==============================] - 3s 2ms/step - loss: 0.3819 - accuracy: 0.8667 - val_loss: 0.3809 - val_accuracy: 0.8612
Epoch 6/50
1719/1719 [==============================] - 2s 1ms/step - loss: 0.3715 - accuracy: 0.8667 - val_loss: 0.3648 - val_accuracy: 0.8652
Epoch 7/50
1719/1719 [==============================] - 2s 1ms/step - loss: 0.3681 - accuracy: 0.8691 - val_loss: 0.3719 - val_accuracy:

In [14]:
# Exponential scheduling
lr = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.01, decay_steps=steps, decay_rate=0.1
)
optimizer = keras.optimizers.SGD(learning_rate=lr)
exp_exponential = Experiment(
    model=model, optimizer=optimizer, epochs=epochs, callbacks=[]
)
history_exponential = exp_exponential.fit(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

Epoch 1/50
1719/1719 [==============================] - 3s 2ms/step - loss: 0.7730 - accuracy: 0.7366 - val_loss: 0.4554 - val_accuracy: 0.8350
Epoch 2/50
1719/1719 [==============================] - 2s 1ms/step - loss: 0.4610 - accuracy: 0.8381 - val_loss: 0.4246 - val_accuracy: 0.8472
Epoch 3/50
1719/1719 [==============================] - 2s 1ms/step - loss: 0.4283 - accuracy: 0.8480 - val_loss: 0.4148 - val_accuracy: 0.8446
Epoch 4/50
1719/1719 [==============================] - 2s 1ms/step - loss: 0.4099 - accuracy: 0.8547 - val_loss: 0.3962 - val_accuracy: 0.8552
Epoch 5/50
1719/1719 [==============================] - 2s 1ms/step - loss: 0.3905 - accuracy: 0.8613 - val_loss: 0.3883 - val_accuracy: 0.8616
Epoch 6/50
1719/1719 [==============================] - 3s 2ms/step - loss: 0.3790 - accuracy: 0.8648 - val_loss: 0.3888 - val_accuracy: 0.8580
Epoch 7/50
1719/1719 [==============================] - 2s 1ms/step - loss: 0.3703 - accuracy: 0.8680 - val_loss: 0.3872 - val_accuracy:

In [15]:
# Performance scheduling
lr_scheduler = keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
exp_performance = Experiment(
    model=model,
    optimizer=keras.optimizers.SGD(),
    epochs=epochs,
    callbacks=[lr_scheduler],
)
history_performance = exp_performance.fit(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

Epoch 1/50
1719/1719 [==============================] - 3s 2ms/step - loss: 0.7631 - accuracy: 0.7387 - val_loss: 0.4574 - val_accuracy: 0.8324
Epoch 2/50
1719/1719 [==============================] - 3s 1ms/step - loss: 0.4541 - accuracy: 0.8395 - val_loss: 0.4605 - val_accuracy: 0.8290
Epoch 3/50
1719/1719 [==============================] - 2s 1ms/step - loss: 0.4259 - accuracy: 0.8505 - val_loss: 0.4058 - val_accuracy: 0.8506
Epoch 4/50
1719/1719 [==============================] - 3s 1ms/step - loss: 0.4007 - accuracy: 0.8582 - val_loss: 0.3954 - val_accuracy: 0.8544
Epoch 5/50
1719/1719 [==============================] - 2s 1ms/step - loss: 0.3848 - accuracy: 0.8635 - val_loss: 0.3813 - val_accuracy: 0.8572
Epoch 6/50
1719/1719 [==============================] - 2s 1ms/step - loss: 0.3690 - accuracy: 0.8689 - val_loss: 0.3747 - val_accuracy: 0.8632
Epoch 7/50
1719/1719 [==============================] - 2s 1ms/step - loss: 0.3689 - accuracy: 0.8691 - val_loss: 0.3793 - val_accuracy: